In [13]:
from langchain_core.documents import Document
from transformers import CLIPProcessor,CLIPModel
from PIL import Image
import torch
import numpy as np 

from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.schema.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.vectorstores import FAISS


In [14]:
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [15]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [16]:
def embed_image(image_data):
    """Embed image Using Clip"""
    
    if isinstance(image_data,str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data
        
    inputs =  clip_processor(images=image,return_tensor = "pt")     
    
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        
        features = features / features.norm(dim = 1 , keepdim = True)   
        return features.squeeze().numpy()
    
    
def embed_text(text):
    """Embed using Clip """
    
    inputs = clip_processor(
        text=text,
        return_tensor = "pt",
        padding = True,
        truncation = True,
        max_length = 77
    )
    with torch.no_grad():
       features = clip_model.get_text_features(**inputs)
       features = features / features.norm(dim = 1 , keepdim = True)   
       return features.squeeze().numpy()    
       

In [17]:
import fitz
pdf_path = "multi_modal_sample.pdf"
doc = fitz.open(pdf_path)

all_docs=[]
all_embeddings = []
image_data_store = {}

splitter = RecursiveCharacterTextSplitter(chunk_size = 500 , chunk_overlap =100)


In [ ]:
import io
import base64
from PIL import Image
from langchain_core.documents import Document

all_embeddings = []
all_docs = []

for i, page in enumerate(doc):

    # =========================
    # TEXT PROCESSING
    # =========================

    text = page.get_text()

    if text.strip():

        temp_doc = Document(
            page_content=text,
            metadata={
                "page": i,
                "type": "text"
            }
        )

        text_chunks = splitter.split_documents([temp_doc])

        for chunk in text_chunks:

            embedding = embed_text(chunk.page_content)

            all_embeddings.append(embedding)
            all_docs.append(chunk)


    # =========================
    # IMAGE PROCESSING
    # =========================

    for img_index, img in enumerate(page.get_images(full=True)):

        try:

            # Image ka XREF
            xref = img[0]

            # PDF se image extract
            base_image = doc.extract_image(xref)

            # Image bytes
            image_bytes = base_image["image"]

            # PIL image
            pil_image = Image.open(
                io.BytesIO(image_bytes)
            ).convert("RGB")

            # Image ko base64 me convert karna
            buffered = io.BytesIO()

            pil_image.save(
                buffered,
                format="JPEG"
            )

            img_base64 = base64.b64encode(
                buffered.getvalue()
            ).decode("utf-8")

            # Image metadata
            image_doc = Document(
                page_content=img_base64,
                metadata={
                    "page": i,
                    "type": "image",
                    "image_index": img_index,
                    "format": "jpeg"
                }
            )

            # Image document ko store karna
            all_docs.append(image_doc)

            print(
                f"Image extracted → Page: {i}, "
                f"Image: {img_index}"
            )

        except Exception as e:

            print(
                f"Error extracting image "
                f"from page {i}: {e}"
            )   
            continue 
doc.close()            

AttributeError: 'list' object has no attribute 'page_content'